In [1]:
import csv
import heapq
import json
import time
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from PIL import Image
import timm
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms

import wandb
import mlflow
import mlflow.pytorch
from tqdm import tqdm
from torch.amp import GradScaler, autocast
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for saving figs
from torchinfo import summary
import numpy as np
from collections import Counter
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

/home/lang-chain/Documents/Weed-Detection-deep-weeds-/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Environment setup
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"CUDA Available: {torch.cuda.is_available()}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

CUDA Available: True
Device : cuda
GPU    : NVIDIA GeForce RTX 3060 Laptop GPU


In [3]:
# path
csv_path = '../artifacts/data_ingestion/normalized/v_20260307_192121_5a0b6671da93a189/'

In [4]:
train_images = os.path.join(csv_path, 'images', 'train')
test_images  = os.path.join(csv_path, 'images', 'test')
val_images   = os.path.join(csv_path, 'images', 'val')

train_csv = os.path.join(csv_path, 'labels', 'train.csv')
test_csv  = os.path.join(csv_path, 'labels', 'test.csv')
val_csv   = os.path.join(csv_path, 'labels', 'val.csv')

In [5]:
# Checkpoint directories
CHECKPOINTS_DIR  = Path('checkpoints')
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH  = Path('best_model.pth')
FINAL_MODEL_PATH = Path('final_model.pth')
HISTORY_PATH     = Path('training_history.json')

In [6]:
# ── Model Config ───────────────────────────────────────────────────────────────
ARCHITECTURE   = 'efficientnet_b3'
NUM_CLASSES    = 9
PRETRAINED     = True
DROPOUT_RATE   = 0.3
INPUT_SIZE     = 256
MIXED_PRECISION= True

# ── Training Config ────────────────────────────────────────────────────────────
BATCH_SIZE     = 16
NUM_WORKERS    = 4
PIN_MEMORY     = True
LEARNING_RATE  = 1e-4
WEIGHT_DECAY   = 1e-4
LR_SCHEDULER   = 'cosine'     # cosine | plateau
WARMUP_EPOCHS  = 3
EPOCHS         = 30
LABEL_SMOOTHING= 0.1
SAMPLER        = 'weighted'   # weighted | random
SAVE_TOP_K     = 3
EARLY_STOP_PAT = 5
MONITOR_METRIC = 'val_acc'    # val_acc | val_loss

# ── Focal Loss ─────────────────────────────────────────────────────────────────
USE_FOCAL_LOSS = True
FOCAL_GAMMA    = 2.0

# ── Data Config ────────────────────────────────────────────────────────────────
IMAGENET_MEAN  = [0.485, 0.456, 0.406]
IMAGENET_STD   = [0.229, 0.224, 0.225]

In [7]:
SPECIES_MAP = {
    0: 'Chinee Apple', 1: 'Lantana', 2: 'Parkinsonia',
    3: 'Parthenium', 4: 'Prickly acacia', 5: 'Rubber Vine',
    6: 'Siam weed', 7: 'Snake weed', 8: 'Negative',
}
CLASS_NAMES = [SPECIES_MAP[i] for i in range(NUM_CLASSES)]

In [8]:
# ── MLflow Config ──────────────────────────────────────────────────────────────
MLFLOW_TRACKING_URI = 'sqlite:///mlflow.db'
MLFLOW_EXPERIMENT   = 'weed-detection'

# ── W&B Config ─────────────────────────────────────────────────────────────────
WANDB_PROJECT = 'weed-detection'

In [9]:
# ── Run ID ─────────────────────────────────────────────────────────────────────
VERSION_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_NAME   = f'{ARCHITECTURE}_{VERSION_ID}'

print(f'Version ID : {VERSION_ID}')
print(f'Run name   : {RUN_NAME}')

Version ID : 20260310_145145
Run name   : efficientnet_b3_20260310_145145


In [10]:
# ── Focal Loss ─────────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    """
    Focal Loss — down-weights easy examples so the model focuses on hard ones.
    Works with class weights to further handle imbalance.
    """
    def __init__(self, weight=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.weight    = weight
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # inputs: float32 logits  (always — caller must cast from float16)
        # targets: long labels
        ce_loss    = F.cross_entropy(inputs, targets, reduction='none', weight=self.weight)
        pt         = torch.exp(-ce_loss)                 # pt = e^(-ce) = p_correct
        focal_loss = (1 - pt) ** self.gamma * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


# ── Utilities ──────────────────────────────────────────────────────────────────
def get_folder_size(folder):
    total = 0
    for path, dirs, files in os.walk(folder):
        for f in files:
            total += os.path.getsize(os.path.join(path, f))
    return total / (1024 ** 2)


def file_size_mb(path):
    return os.path.getsize(path) / (1024 ** 2)

In [11]:
def load_samples(csv_path: Path, images_dir: Path):
    """Load (image_path, label) pairs from CSV. Skips missing files."""
    samples = []
    missing = 0

    with open(csv_path, 'r') as f:
        reader = csv.DictReader(f)
        for row in tqdm(reader, desc=f"Loading {Path(csv_path).stem}"):
            filename   = row['Filename']
            label      = int(row['Label'])
            image_path = os.path.join(images_dir, filename)

            if os.path.exists(image_path):
                samples.append((image_path, label))
            else:
                missing += 1

    print(f'  Loaded  : {len(samples)} samples  |  Missing: {missing}')
    return samples

In [12]:
print('Train:')
train_samples = load_samples(train_csv, train_images)
print('Val:')
val_samples   = load_samples(val_csv,   val_images)

print(f'\nFirst 3 train : {train_samples[:3]}')

Train:


Loading train: 52525it [00:00, 136105.51it/s]


  Loaded  : 52525 samples  |  Missing: 0
Val:


Loading val: 17511it [00:00, 140852.86it/s]

  Loaded  : 17511 samples  |  Missing: 0

First 3 train : [('../artifacts/data_ingestion/normalized/v_20260307_192121_5a0b6671da93a189/images/train/20171109-175921-2.jpg', 5), ('../artifacts/data_ingestion/normalized/v_20260307_192121_5a0b6671da93a189/images/train/20170714-142019-3.jpg', 1), ('../artifacts/data_ingestion/normalized/v_20260307_192121_5a0b6671da93a189/images/train/20170718-101402-2.jpg', 0)]


In [13]:
# ── Class weights ──────────────────────────────────────────────────────────────
train_labels  = [label for _, label in train_samples]
label_counts  = Counter(train_labels)
total_samples = len(train_labels)

weight_exponent = 0.5   # 0.5 >1 = more aggressive minority-class boosting
# class_weights   = [
#     (total_samples / (NUM_CLASSES * label_counts[c])) ** weight_exponent
#     if label_counts[c] > 0 else 1.0
#     for c in range(NUM_CLASSES)
# ]
class_weights = [
    total_samples / (NUM_CLASSES * label_counts[c])
    for c in range(NUM_CLASSES)
]

print('\nSamples per class:')
for c in range(NUM_CLASSES):
    print(f'  {c} {SPECIES_MAP[c]:<18} : {label_counts[c]:>5}  weight={class_weights[c]:.4f}')


Samples per class:
  0 Chinee Apple       :  3379  weight=1.7272
  1 Lantana            :  3188  weight=1.8306
  2 Parkinsonia        :  3094  weight=1.8863
  3 Parthenium         :  3066  weight=1.9035
  4 Prickly acacia     :  3185  weight=1.8324
  5 Rubber Vine        :  3026  weight=1.9287
  6 Siam weed          :  3221  weight=1.8119
  7 Snake weed         :  3049  weight=1.9141
  8 Negative           : 27317  weight=0.2136


In [14]:
# ── Dataset ────────────────────────────────────────────────────────────────────
class DeepWeedDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform
        self.labels    = [label for _, label in samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

In [15]:
# ── Transforms ─────────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=90),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.RandomResizedCrop(size=INPUT_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.CenterCrop(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [16]:
# ── DataLoaders ────────────────────────────────────────────────────────────────
train_dataset  = DeepWeedDataset(train_samples, train_transform)
val_dataset    = DeepWeedDataset(val_samples,   val_transform)

sample_weights = [class_weights[label] for label in train_dataset.labels]
sampler        = WeightedRandomSampler(
    weights    = sample_weights,
    num_samples= len(sample_weights),
    replacement= True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size = BATCH_SIZE,
    sampler    = sampler,
    num_workers= NUM_WORKERS,
    pin_memory = PIN_MEMORY,
    drop_last  = True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size = BATCH_SIZE,
    shuffle    = False,
    num_workers= NUM_WORKERS,
    pin_memory = PIN_MEMORY,
)

print(f'Train batches : {len(train_loader)}  (batch={BATCH_SIZE}, sampler={SAMPLER})')
print(f'Val batches   : {len(val_loader)}')

Train batches : 3282  (batch=16, sampler=weighted)
Val batches   : 1095


In [17]:
# Sanity check batch shape
imgs, lbls = next(iter(train_loader))
print(f'Batch shape : {imgs.shape}  labels={lbls[:8].tolist()}')

Batch shape : torch.Size([16, 3, 256, 256])  labels=[0, 8, 1, 1, 5, 5, 7, 6]


In [18]:
# ── Model ──────────────────────────────────────────────────────────────────────
scaler = GradScaler('cuda') if MIXED_PRECISION else None

model = timm.create_model(
    ARCHITECTURE,
    pretrained  = PRETRAINED,
    num_classes = 0,
    global_pool = 'avg'
)

num_features = model.num_features
print(f'Backbone output features : {num_features}')

# Custom classifier head
model.classifier = nn.Sequential(
    nn.Dropout(p=DROPOUT_RATE),
    nn.Linear(num_features, NUM_CLASSES),
)

model = model.to(DEVICE)

# Parameter count
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Architecture     : {ARCHITECTURE}')
print(f'Total params     : {total_params:,}')
print(f'Trainable params : {trainable_params:,}')

Backbone output features : 1536
Architecture     : efficientnet_b3
Total params     : 10,710,065
Trainable params : 10,710,065


In [19]:
# Forward pass sanity check
model.eval()
with torch.no_grad():
    dummy = torch.randn(2, 3, INPUT_SIZE, INPUT_SIZE).to(DEVICE)
    if MIXED_PRECISION:
        with autocast('cuda'):
            out = model(dummy)
        out = out.float()   # cast to float32 — same as training fix
    else:
        out = model(dummy)

print(f'Forward pass OK : input={dummy.shape} → output={out.shape}  dtype={out.dtype}')

Forward pass OK : input=torch.Size([2, 3, 256, 256]) → output=torch.Size([2, 9])  dtype=torch.float32


In [20]:
# Model summary
print("\nModel Summary:")
summary(
    model,
    input_size=(1, 3, INPUT_SIZE, INPUT_SIZE),
    col_names=["input_size", "output_size", "num_params", "trainable"],
    depth=3
)


Model Summary:


Layer (type:depth-idx)                        Input Shape               Output Shape              Param #                   Trainable
EfficientNet                                  [1, 3, 256, 256]          [1, 9]                    --                        True
├─Conv2d: 1-1                                 [1, 3, 256, 256]          [1, 40, 128, 128]         1,080                     True
├─BatchNormAct2d: 1-2                         [1, 40, 128, 128]         [1, 40, 128, 128]         80                        True
│    └─Identity: 2-1                          [1, 40, 128, 128]         [1, 40, 128, 128]         --                        --
│    └─SiLU: 2-2                              [1, 40, 128, 128]         [1, 40, 128, 128]         --                        --
├─Sequential: 1-3                             [1, 40, 128, 128]         [1, 384, 8, 8]            --                        True
│    └─Sequential: 2-3                        [1, 40, 128, 128]         [1, 24, 128, 128]       

In [21]:
# Model size
MODEL_TEMP_PATH = "temp_model.pth"
torch.save(model.state_dict(), MODEL_TEMP_PATH)
print(f"Model size : {os.path.getsize(MODEL_TEMP_PATH) / (1024**2):.2f} MB")
os.remove(MODEL_TEMP_PATH)

Model size : 41.38 MB


In [22]:
# ── Loss / Optimizer / Scheduler ───────────────────────────────────────────────
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

if USE_FOCAL_LOSS:
    criterion = FocalLoss(weight=class_weights_tensor, gamma=FOCAL_GAMMA)
    print(f'Loss      : FocalLoss  gamma={FOCAL_GAMMA}')
else:
    criterion = nn.CrossEntropyLoss(
        weight         = class_weights_tensor,
        label_smoothing= LABEL_SMOOTHING,
    )
    print(f'Loss      : CrossEntropyLoss  smoothing={LABEL_SMOOTHING}')

optimizer = optim.AdamW(
    model.parameters(),
    lr          = LEARNING_RATE,
    weight_decay= WEIGHT_DECAY,
)
print(f'Optimizer : AdamW  lr={LEARNING_RATE}  wd={WEIGHT_DECAY}')

effective_epochs = max(EPOCHS - WARMUP_EPOCHS, 1)
scheduler = CosineAnnealingLR(optimizer, T_max=effective_epochs, eta_min=1e-6)
print(f'Scheduler : CosineAnnealingLR  T_max={effective_epochs}  eta_min=1e-6')
print(f'Warmup    : {WARMUP_EPOCHS} epochs (linear LR ramp)')

Loss      : FocalLoss  gamma=2.0
Optimizer : AdamW  lr=0.0001  wd=0.0001
Scheduler : CosineAnnealingLR  T_max=27  eta_min=1e-6
Warmup    : 3 epochs (linear LR ramp)


In [23]:
# ── MLflow setup ───────────────────────────────────────────────────────────────
print("\nSetting up MLflow and W&B...")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

mlflow_run = mlflow.start_run(run_name=RUN_NAME)


Setting up MLflow and W&B...


2026/03/10 14:52:02 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/10 14:52:03 INFO mlflow.store.db.utils: Updating database tables
2026/03/10 14:52:04 INFO mlflow.tracking.fluent: Experiment with name 'weed-detection' does not exist. Creating a new experiment.


In [24]:
# Log all hyperparameters to MLflow
mlflow.log_params({
    'architecture'   : ARCHITECTURE,
    'pretrained'     : PRETRAINED,
    'num_classes'    : NUM_CLASSES,
    'input_size'     : INPUT_SIZE,
    'dropout_rate'   : DROPOUT_RATE,
    'epochs'         : EPOCHS,
    'batch_size'     : BATCH_SIZE,
    'learning_rate'  : LEARNING_RATE,
    'weight_decay'   : WEIGHT_DECAY,
    'lr_scheduler'   : LR_SCHEDULER,
    'warmup_epochs'  : WARMUP_EPOCHS,
    'label_smoothing': LABEL_SMOOTHING,
    'sampler'        : SAMPLER,
    'early_stop_pat' : EARLY_STOP_PAT,
    'monitor_metric' : MONITOR_METRIC,
    'total_params'   : total_params,
    'trainable_params': trainable_params,
    'device'         : str(DEVICE),
    'num_train'      : len(train_samples),
    'num_val'        : len(val_samples),
    'version_id'     : VERSION_ID,
    'use_focal_loss' : USE_FOCAL_LOSS,
    'focal_gamma'    : FOCAL_GAMMA if USE_FOCAL_LOSS else None,
    'weight_exponent': weight_exponent,
    'mixed_precision': MIXED_PRECISION,
})

mlflow.set_tags({
    'model_family': 'efficientnet',
    'dataset'     : 'deepweeds',
    'task'        : 'multiclass_classification',
})

print(f'MLflow run ID : {mlflow_run.info.run_id}')
print(f'MLflow UI     : http://localhost:5000')

MLflow run ID : b94ef28ac31e47ed8d17cb762c9d653a
MLflow UI     : http://localhost:5000


In [25]:
# ── W&B setup ──────────────────────────────────────────────────────────────────
wandb_run = wandb.init(
    project = WANDB_PROJECT,
    name    = RUN_NAME,
    config  = {
        'architecture'   : ARCHITECTURE,
        'pretrained'     : PRETRAINED,
        'num_classes'    : NUM_CLASSES,
        'input_size'     : INPUT_SIZE,
        'dropout_rate'   : DROPOUT_RATE,
        'epochs'         : EPOCHS,
        'batch_size'     : BATCH_SIZE,
        'learning_rate'  : LEARNING_RATE,
        'weight_decay'   : WEIGHT_DECAY,
        'lr_scheduler'   : LR_SCHEDULER,
        'warmup_epochs'  : WARMUP_EPOCHS,
        'label_smoothing': LABEL_SMOOTHING,
        'sampler'        : SAMPLER,
        'num_train'      : len(train_samples),
        'num_val'        : len(val_samples),
        'use_focal_loss' : USE_FOCAL_LOSS,
        'focal_gamma'    : FOCAL_GAMMA if USE_FOCAL_LOSS else None,
        'weight_exponent': weight_exponent,
        'mixed_precision': MIXED_PRECISION,
    },
    tags = ['training', ARCHITECTURE, 'deepweeds'],
)

# Watch model: log gradients + weights every 100 steps
wandb.watch(model, log='all', log_freq=100)

# Cross-link MLflow ↔ W&B
mlflow.set_tag('wandb_url',    wandb_run.url)
mlflow.set_tag('wandb_run_id', wandb_run.id)

# Log class weight table to W&B
cw_table = wandb.Table(
    columns = ['class_id', 'species', 'count', 'weight'],
    data    = [
        [c, SPECIES_MAP[c], label_counts[c], round(class_weights[c], 4)]
        for c in range(NUM_CLASSES)
    ]
)
wandb.log({'class_weights': cw_table})

# Log class weight table to MLflow
for c in range(NUM_CLASSES):
    mlflow.log_metric(f'class_count_{SPECIES_MAP[c]}',  label_counts[c])
    mlflow.log_metric(f'class_weight_{SPECIES_MAP[c]}', class_weights[c])

print(f'W&B run URL : {wandb_run.url}')

# cuDNN benchmark for speed
torch.backends.cudnn.benchmark = True

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: kumardahal788 (kumardahal788-loyalist-college-of-toronto) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B run URL : https://wandb.ai/kumardahal788-loyalist-college-of-toronto/weed-detection/runs/jhu97y4d


In [26]:
# ── Helper: confusion matrix figure ───────────────────────────────────────────
def plot_confusion_matrix(all_labels, all_preds, class_names, epoch):
    """
    Build and return a confusion matrix figure.
    Logged to both W&B and MLflow as an artifact.
    """
    cm   = confusion_matrix(all_labels, all_preds, labels=list(range(len(class_names))))
    cm_n = cm.astype('float') / (cm.sum(axis=1, keepdims=True) + 1e-8)  # row-normalize

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        cm_n, annot=True, fmt='.2f', cmap='Blues',
        xticklabels=class_names, yticklabels=class_names, ax=ax
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix — Epoch {epoch}')
    plt.tight_layout()
    return fig


# ── Helper: per-class accuracy table for W&B ──────────────────────────────────
def make_class_acc_table(epoch_cls_acc):
    return wandb.Table(
        columns = ['class_id', 'species', 'val_acc'],
        data    = [
            [c, SPECIES_MAP[c], round(epoch_cls_acc.get(c, 0.0), 4)]
            for c in range(NUM_CLASSES)
        ]
    )

print('Helper functions defined.')

Helper functions defined.


In [27]:
# ── Training state ─────────────────────────────────────────────────────────────
history          = []
best_val_acc     = 0.0
best_val_loss    = float('inf')
best_epoch       = 0
per_class_acc    = {}
no_improve_count = 0
nan_batch_count  = 0   # track total NaN batches across training

training_start   = time.time()
ckpt_heap        = []

print(f"\nStarting training — {EPOCHS} epochs on {DEVICE}")
print("=" * 70)


Starting training — 30 epochs on cuda


In [ ]:
try:
    for epoch in range(1, EPOCHS + 1):
        epoch_start  = time.time()
        epoch_nan_batches = 0

        # ── LR Warmup ────────────────────────────────────────────────────────
        if epoch <= WARMUP_EPOCHS:
            warmup_lr = LEARNING_RATE * epoch / WARMUP_EPOCHS
            for pg in optimizer.param_groups:
                pg["lr"] = warmup_lr

        # ════════════════════════════════════════════════════════════════════
        # TRAIN
        # ════════════════════════════════════════════════════════════════════
        model.train()
        train_loss_sum = 0.0
        train_correct  = 0
        train_total    = 0

        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
        for batch_idx, (images, labels) in enumerate(train_pbar):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            try:
                if MIXED_PRECISION:
                    with autocast('cuda'):
                        outputs = model(images)

                    # ── NaN FIX: cast float16 → float32 before loss ────────
                    outputs = outputs.float()

                    if torch.isnan(outputs).any() or torch.isinf(outputs).any():
                        epoch_nan_batches += 1
                        nan_batch_count   += 1
                        optimizer.zero_grad(set_to_none=True)
                        continue

                    loss = criterion(outputs, labels)

                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                    #torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()

                else:
                    outputs = model(images)
                    if torch.isnan(outputs).any() or torch.isinf(outputs).any():
                        epoch_nan_batches += 1
                        nan_batch_count   += 1
                        optimizer.zero_grad(set_to_none=True)
                        continue

                    loss = criterion(outputs, labels)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                    #torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()

                train_loss_sum += loss.item() * images.size(0)
                preds           = outputs.argmax(dim=1)
                train_correct  += (preds == labels).sum().item()
                train_total    += images.size(0)

                train_pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc' : f'{(preds == labels).float().mean():.4f}'
                })

            except Exception as e:
                print(f"Error in train batch {batch_idx}: {e}")
                optimizer.zero_grad(set_to_none=True)
                continue

        train_loss = train_loss_sum / train_total if train_total > 0 else 0.0
        train_acc  = train_correct  / train_total if train_total > 0 else 0.0

        # ════════════════════════════════════════════════════════════════════
        # VALIDATION
        # ════════════════════════════════════════════════════════════════════
        model.eval()
        val_loss_sum    = 0.0
        val_correct     = 0
        val_total       = 0
        class_correct   = defaultdict(int)
        class_total_c   = defaultdict(int)
        stable_batches  = 0
        all_preds_list  = []   # for confusion matrix
        all_labels_list = []   # for confusion matrix

        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Val]")
        with torch.no_grad():
            for batch_idx, (images, labels) in enumerate(val_pbar):
                images = images.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)

                try:
                    if MIXED_PRECISION:
                        with autocast('cuda'):
                            outputs = model(images)

                        # ── NaN FIX: cast float16 → float32 before loss ────
                        # This is the primary fix for the NaN issue.
                        # float16 max ~65504; large activations overflow → NaN.
                        # float32 max ~3.4e38; no overflow in practice.
                        outputs = outputs.float()

                    else:
                        outputs = model(images)

                    # Check after cast — should rarely trigger now
                    if torch.isnan(outputs).any() or torch.isinf(outputs).any():
                        epoch_nan_batches += 1
                        nan_batch_count   += 1
                        continue

                    loss = criterion(outputs, labels)

                    if torch.isnan(loss).any() or torch.isinf(loss).any():
                        epoch_nan_batches += 1
                        nan_batch_count   += 1
                        continue

                    val_loss_sum  += loss.item() * images.size(0)
                    preds          = outputs.argmax(dim=1)
                    val_correct   += (preds == labels).sum().item()
                    val_total     += images.size(0)
                    stable_batches+= 1

                    all_preds_list .extend(preds.cpu().tolist())
                    all_labels_list.extend(labels.cpu().tolist())

                    for lbl, pred in zip(labels.cpu().tolist(), preds.cpu().tolist()):
                        class_total_c[lbl] += 1
                        class_correct[lbl] += int(lbl == pred)

                    val_pbar.set_postfix({
                        'loss': f'{loss.item():.4f}',
                        'acc' : f'{(preds == labels).float().mean():.4f}'
                    })

                except Exception as e:
                    print(f"Error in val batch {batch_idx}: {e}")
                    continue

        if val_total > 0:
            val_loss = val_loss_sum / val_total
            val_acc  = val_correct  / val_total
        else:
            val_loss = 0.0
            val_acc  = 0.0
            print("⚠️ No stable validation batches — check model/data")

        epoch_cls_acc = {
            cls: round(class_correct[cls] / class_total_c[cls], 4)
            if class_total_c[cls] > 0 else 0.0
            for cls in range(NUM_CLASSES)
        }

        # ── Scheduler step ───────────────────────────────────────────────────
        if epoch > WARMUP_EPOCHS:
            if isinstance(scheduler, ReduceLROnPlateau):
                scheduler.step(val_acc)
            else:
                scheduler.step()

        current_lr = optimizer.param_groups[0]["lr"]
        epoch_time = time.time() - epoch_start

        # ── Console output ───────────────────────────────────────────────────
        print(
            f"\nEpoch [{epoch:>3}/{EPOCHS}] "
            f"train_loss={train_loss:.4f} "
            f"train_acc={train_acc:.4f} "
            f"val_loss={val_loss:.4f} "
            f"val_acc={val_acc:.4f} "
            f"lr={current_lr:.6f} "
            f"time={epoch_time:.1f}s "
            f"stable={stable_batches}/{len(val_loader)} "
            f"nan_batches={epoch_nan_batches}"
        )

        # ── W&B logging ──────────────────────────────────────────────────────
        try:
            log_dict = {
                # Core metrics
                "epoch"                : epoch,
                "train/loss"           : train_loss,
                "train/acc"            : train_acc,
                "val/loss"             : val_loss,
                "val/acc"              : val_acc,
                "lr"                   : current_lr,
                "epoch_time_s"         : epoch_time,
                # Stability tracking
                "debug/stable_val_batches" : stable_batches,
                "debug/nan_batches_epoch"  : epoch_nan_batches,
                "debug/nan_batches_total"  : nan_batch_count,
                # Per-class val accuracy (individual lines on W&B chart)
                **{f"val/cls_acc/{SPECIES_MAP[c]}": acc
                   for c, acc in epoch_cls_acc.items()},
                # Per-class accuracy table
                "val/per_class_table"  : make_class_acc_table(epoch_cls_acc),
            }

            # Confusion matrix every 5 epochs and final epoch
            if (epoch % 5 == 0 or epoch == EPOCHS) and all_labels_list:
                cm_fig = plot_confusion_matrix(
                    all_labels_list, all_preds_list, CLASS_NAMES, epoch
                )
                log_dict["val/confusion_matrix"] = wandb.Image(cm_fig)
                plt.close(cm_fig)

            wandb.log(log_dict)

        except Exception as e:
            print(f"Warning: W&B logging failed — {e}")

        # ── MLflow logging ───────────────────────────────────────────────────
        try:
            mlflow.log_metrics({
                # Core metrics
                "train_loss"          : train_loss,
                "train_acc"           : train_acc,
                "val_loss"            : val_loss,
                "val_acc"             : val_acc,
                "lr"                  : current_lr,
                "epoch_time_s"        : epoch_time,
                # Stability
                "stable_val_batches"  : stable_batches,
                "nan_batches_epoch"   : epoch_nan_batches,
                "nan_batches_total"   : nan_batch_count,
                # Per-class val accuracy
                **{f"cls_acc_{SPECIES_MAP[c]}": acc
                   for c, acc in epoch_cls_acc.items()},
            }, step=epoch)

            # Save confusion matrix as MLflow artifact every 5 epochs
            if (epoch % 5 == 0 or epoch == EPOCHS) and all_labels_list:
                cm_fig = plot_confusion_matrix(
                    all_labels_list, all_preds_list, CLASS_NAMES, epoch
                )
                cm_path = f"confusion_matrix_epoch_{epoch:03d}.png"
                cm_fig.savefig(cm_path, dpi=100, bbox_inches='tight')
                mlflow.log_artifact(cm_path)
                plt.close(cm_fig)
                os.remove(cm_path)

        except Exception as e:
            print(f"Warning: MLflow logging failed — {e}")

        # ── Best model tracking ───────────────────────────────────────────────
        is_best = val_acc > best_val_acc

        if is_best:
            best_val_acc  = val_acc
            best_val_loss = val_loss
            best_epoch    = epoch
            per_class_acc = epoch_cls_acc
            no_improve_count = 0

            torch.save(model.state_dict(), BEST_MODEL_PATH)

            # W&B — log best model as artifact
            try:
                artifact = wandb.Artifact(
                    name     = "best-model",
                    type     = "model",
                    metadata = {"val_acc": best_val_acc, "epoch": best_epoch},
                )
                artifact.add_file(str(BEST_MODEL_PATH))
                wandb.log_artifact(artifact)
                wandb.log({
                    "best/val_acc" : best_val_acc,
                    "best/val_loss": best_val_loss,
                    "best/epoch"   : best_epoch,
                })
            except Exception as e:
                print(f"Warning: W&B artifact logging failed — {e}")

            # MLflow — log best metrics
            try:
                mlflow.log_metrics({
                    "best_val_acc" : best_val_acc,
                    "best_val_loss": best_val_loss,
                    "best_epoch"   : float(best_epoch),
                }, step=epoch)
                mlflow.log_artifact(str(BEST_MODEL_PATH))
            except Exception as e:
                print(f"Warning: MLflow best-model logging failed — {e}")

            print(f"⭐ New best! val_acc={best_val_acc:.4f}")

        else:
            no_improve_count += 1

        # ── Top-K checkpoints ─────────────────────────────────────────────────
        ckpt_path = CHECKPOINTS_DIR / f"epoch_{epoch:03d}_acc_{val_acc:.4f}.pth"
        torch.save(model.state_dict(), ckpt_path)
        heapq.heappush(ckpt_heap, (val_acc, str(ckpt_path)))

        if len(ckpt_heap) > SAVE_TOP_K:
            worst_acc, worst_path = heapq.heappop(ckpt_heap)
            worst = Path(worst_path)
            if worst.exists():
                worst.unlink()

        # ── Save history ──────────────────────────────────────────────────────
        history.append({
            "epoch"             : epoch,
            "train_loss"        : train_loss,
            "train_acc"         : train_acc,
            "val_loss"          : val_loss,
            "val_acc"           : val_acc,
            "lr"                : current_lr,
            "epoch_time_s"      : epoch_time,
            "stable_val_batches": stable_batches,
            "nan_batches"       : epoch_nan_batches,
        })

        with open(HISTORY_PATH, "w") as f:
            json.dump(history, f, indent=2)

        # ── Early stopping ────────────────────────────────────────────────────
        if no_improve_count >= EARLY_STOP_PAT:
            print(
                f"\n🛑 Early stopping at epoch {epoch} "
                f"(no improvement for {EARLY_STOP_PAT} epochs)"
            )
            try:
                mlflow.set_tag("stopped_early", "True")
                mlflow.log_metric("stopped_at_epoch", float(epoch))
            except:
                pass
            break

except KeyboardInterrupt:
    print("\n🛑 Training interrupted by user. Saving checkpoint...")
    interrupted_path = CHECKPOINTS_DIR / f"interrupted_epoch_{epoch}_acc_{val_acc:.4f}.pth"
    torch.save(model.state_dict(), interrupted_path)
    print(f"Checkpoint saved to {interrupted_path}")

finally:
    try:
        torch.save(model.state_dict(), FINAL_MODEL_PATH)
        print(f"\nFinal model saved to {FINAL_MODEL_PATH}")

        total_time = time.time() - training_start
        print("=" * 70)
        print(f"Training complete : {len(history)} epochs")
        print(f"Total time        : {total_time / 60:.1f} minutes")
        print(f"Best val_acc      : {best_val_acc:.4f} at epoch {best_epoch}")
        print(f"Total NaN batches : {nan_batch_count}")

        # ── Final classification report ───────────────────────────────────────
        if all_labels_list and all_preds_list:
            report = classification_report(
                all_labels_list, all_preds_list,
                target_names=CLASS_NAMES, output_dict=True
            )
            print("\nFinal classification report:")
            print(classification_report(
                all_labels_list, all_preds_list,
                target_names=CLASS_NAMES
            ))

            # Log per-class F1 to both trackers
            for cls_name in CLASS_NAMES:
                if cls_name in report:
                    try:
                        mlflow.log_metric(f"final_f1_{cls_name}",        report[cls_name]['f1-score'])
                        mlflow.log_metric(f"final_precision_{cls_name}", report[cls_name]['precision'])
                        mlflow.log_metric(f"final_recall_{cls_name}",    report[cls_name]['recall'])
                    except:
                        pass

            try:
                wandb.log({
                    "final/macro_f1"      : report['macro avg']['f1-score'],
                    "final/weighted_f1"   : report['weighted avg']['f1-score'],
                    "final/best_val_acc"  : best_val_acc,
                    "final/best_epoch"    : best_epoch,
                    "final/total_nan_batches": nan_batch_count,
                })
            except:
                pass

        # ── Log final summary to MLflow ───────────────────────────────────────
        mlflow.log_metrics({
            "final_train_loss"           : history[-1]['train_loss'] if history else 0,
            "final_val_acc"              : history[-1]['val_acc']    if history else 0,
            "total_training_time_min"    : total_time / 60,
            "total_epochs_completed"     : float(len(history)),
            "total_nan_batches"          : float(nan_batch_count),
        })
        mlflow.log_artifact(str(HISTORY_PATH))
        mlflow.log_artifact(str(FINAL_MODEL_PATH))

        # ── Close W&B ─────────────────────────────────────────────────────────
        if wandb.run is not None:
            wandb.finish()

        # ── End MLflow run ────────────────────────────────────────────────────
        mlflow.end_run()

    except Exception as e:
        print(f"Error during final cleanup: {e}")

Epoch 1/30 [Val]: 100%|██████████| 1095/1095 [00:48<00:00, 22.48it/s, loss=0.0440, acc=0.2857]



Epoch [  1/30] train_loss=0.6911 train_acc=0.7414 val_loss=0.2220 val_acc=0.4722 lr=0.000033 time=504.0s stable=1095/1095 nan_batches=0
⭐ New best! val_acc=0.4722


Epoch 2/30 [Val]: 100%|██████████| 1095/1095 [00:49<00:00, 21.92it/s, loss=0.0142, acc=0.7143] 



Epoch [  2/30] train_loss=0.1764 train_acc=0.8797 val_loss=0.3802 val_acc=0.6821 lr=0.000067 time=480.5s stable=1093/1095 nan_batches=2
⭐ New best! val_acc=0.6821


Epoch 3/30 [Val]: 100%|██████████| 1095/1095 [00:38<00:00, 28.27it/s, loss=0.0065, acc=0.8571] 



Epoch [  3/30] train_loss=0.0966 train_acc=0.9278 val_loss=1.0151 val_acc=0.8210 lr=0.000100 time=489.6s stable=1067/1095 nan_batches=28
⭐ New best! val_acc=0.8210


Epoch 4/30 [Val]: 100%|██████████| 1095/1095 [00:44<00:00, 24.34it/s, loss=0.0004, acc=1.0000]  



Epoch [  4/30] train_loss=0.0560 train_acc=0.9512 val_loss=0.8069 val_acc=0.8264 lr=0.000100 time=461.3s stable=1091/1095 nan_batches=4
⭐ New best! val_acc=0.8264


Epoch 5/30 [Val]: 100%|██████████| 1095/1095 [00:39<00:00, 27.88it/s, loss=0.0049, acc=0.8571]



Epoch [  5/30] train_loss=0.0345 train_acc=0.9653 val_loss=0.0556 val_acc=0.8577 lr=0.000099 time=583.7s stable=1095/1095 nan_batches=0


In [ ]:
out

In [ ]:
"""
Cross-Reference MLflow and W&B Results
"""

import mlflow
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlflow.tracking import MlflowClient
import wandb

# ------------------------- MLflow Analysis -------------------------
print("="*50)
print("MLFLOW ANALYSIS")
print("="*50)

# Set tracking URI
mlflow.set_tracking_uri('sqlite:///mlflow.db')

# Get all runs
experiment = mlflow.get_experiment_by_name("weed-detection")
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# Display runs summary
print("\n📊 All Runs Summary:")
display_cols = ['run_id', 'metrics.val_acc', 'metrics.val_loss', 
                'metrics.train_acc', 'metrics.train_loss',
                'params.learning_rate', 'params.architecture',
                'params.use_focal_loss', 'tags.mlflow.runName']
available_cols = [col for col in display_cols if col in runs_df.columns]
print(runs_df[available_cols].to_string())

# Find best run by validation accuracy
best_run = runs_df.loc[runs_df['metrics.val_acc'].idxmax()]
print(f"\n🏆 Best Run (by val_acc):")
print(f"Run ID: {best_run['run_id']}")
print(f"Run Name: {best_run.get('tags.mlflow.runName', 'N/A')}")
print(f"Val Acc: {best_run['metrics.val_acc']:.4f}")
print(f"Val Loss: {best_run['metrics.val_loss']:.4f}")
print(f"Train Acc: {best_run['metrics.train_acc']:.4f}")
print(f"Learning Rate: {best_run['params.learning_rate']}")

# Plot comparison of runs
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Validation Accuracy comparison
ax1 = axes[0, 0]
runs_df_sorted = runs_df.sort_values('metrics.val_acc', ascending=False)
ax1.barh(range(len(runs_df_sorted)), runs_df_sorted['metrics.val_acc'].values)
ax1.set_yticks(range(len(runs_df_sorted)))
ax1.set_yticklabels(runs_df_sorted['tags.mlflow.runName'].values if 'tags.mlflow.runName' in runs_df_sorted.columns else runs_df_sorted['run_id'].str[:8])
ax1.set_xlabel('Validation Accuracy')
ax1.set_title('MLflow: Validation Accuracy by Run')

# Validation Loss comparison
ax2 = axes[0, 1]
ax2.barh(range(len(runs_df_sorted)), runs_df_sorted['metrics.val_loss'].values)
ax2.set_yticks(range(len(runs_df_sorted)))
ax2.set_yticklabels(runs_df_sorted['tags.mlflow.runName'].values if 'tags.mlflow.runName' in runs_df_sorted.columns else runs_df_sorted['run_id'].str[:8])
ax2.set_xlabel('Validation Loss')
ax2.set_title('MLflow: Validation Loss by Run')

# Scatter plot: val_acc vs val_loss
ax3 = axes[1, 0]
scatter = ax3.scatter(runs_df['metrics.val_loss'], runs_df['metrics.val_acc'],
                      c=range(len(runs_df)), cmap='viridis', s=100)
ax3.set_xlabel('Validation Loss')
ax3.set_ylabel('Validation Accuracy')
ax3.set_title('MLflow: Val Acc vs Val Loss')
plt.colorbar(scatter, ax=ax3, label='Run Index')

# Learning Rate vs Val Acc
ax4 = axes[1, 1]
if 'params.learning_rate' in runs_df.columns:
    runs_df['params.learning_rate'] = pd.to_numeric(runs_df['params.learning_rate'], errors='coerce')
    ax4.scatter(runs_df['params.learning_rate'], runs_df['metrics.val_acc'], s=100)
    ax4.set_xlabel('Learning Rate')
    ax4.set_ylabel('Validation Accuracy')
    ax4.set_title('MLflow: Learning Rate vs Val Acc')
    ax4.set_xscale('log')

plt.tight_layout()
plt.savefig('mlflow_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# ------------------------- Per-Class Performance Analysis -------------------------
print("\n📈 Per-Class Performance Analysis:")
class_cols = [col for col in runs_df.columns if col.startswith('metrics.cls_acc_')]
if class_cols:
    class_data = runs_df[['tags.mlflow.runName'] + class_cols].copy()
    class_data.columns = [col.replace('metrics.cls_acc_', '') if col.startswith('metrics.cls_acc_') else col 
                          for col in class_data.columns]
    
    # Melt for easier plotting
    class_melted = class_data.melt(id_vars=['tags.mlflow.runName'], 
                                    var_name='Class', value_name='Accuracy')
    
    plt.figure(figsize=(12, 6))
    sns.barplot(data=class_melted, x='Class', y='Accuracy', hue='tags.mlflow.runName')
    plt.title('Per-Class Accuracy Across Runs')
    plt.xticks(rotation=45)
    plt.legend(title='Run Name', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig('per_class_accuracy.png', dpi=150, bbox_inches='tight')
    plt.show()

# ------------------------- W&B Integration -------------------------
print("\n" + "="*50)
print("W&B CROSS-REFERENCE")
print("="*50)

# You can also query W&B API if you have wandb installed and logged in
try:
    api = wandb.Api()
    
    # Get your project runs
    runs = api.runs("kumardahal788-loyalist-college-of-toronto/weed-detection")
    
    print("\n📊 W&B Runs Summary:")
    for run in runs:
        print(f"\nRun: {run.name}")
        print(f"  State: {run.state}")
        print(f"  Val Acc: {run.summary.get('val/acc', 'N/A')}")
        print(f"  Train Acc: {run.summary.get('train/acc', 'N/A')}")
        print(f"  URL: {run.url}")
        
        # Find matching MLflow run
        mlflow_match = runs_df[runs_df['tags.wandb_run_id'] == run.id]
        if not mlflow_match.empty:
            print(f"  ✅ MLflow Run ID: {mlflow_match.iloc[0]['run_id']}")
        else:
            print(f"  ❌ No matching MLflow run found")
            
except Exception as e:
    print(f"W&B API error: {e}")
    print("Make sure you're logged in to wandb: wandb login")

# ------------------------- Generate Comparison Report -------------------------
print("\n" + "="*50)
print("COMPARISON REPORT")
print("="*50)

report = {
    'total_runs': len(runs_df),
    'best_val_acc': runs_df['metrics.val_acc'].max(),
    'best_val_acc_run': runs_df.loc[runs_df['metrics.val_acc'].idxmax(), 'tags.mlflow.runName'] 
                        if 'tags.mlflow.runName' in runs_df.columns else 'N/A',
    'avg_val_acc': runs_df['metrics.val_acc'].mean(),
    'std_val_acc': runs_df['metrics.val_acc'].std(),
    'best_val_loss': runs_df['metrics.val_loss'].min(),
    'avg_val_loss': runs_df['metrics.val_loss'].mean(),
}

print("\n📋 Summary Statistics:")
for key, value in report.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

# Save report
import json
with open('training_comparison_report.json', 'w') as f:
    json.dump({k: float(v) if isinstance(v, (float, int)) else v for k, v in report.items()}, f, indent=2)

print("\n✅ Report saved to training_comparison_report.json")
print("📊 MLflow plots saved to mlflow_analysis.png")
print("📈 Per-class accuracy plot saved to per_class_accuracy.png")